In [2]:
!pip install -qU langchain
!pip install -qU langchain-google-genai
from langchain.chat_models import init_chat_model

from google.colab import userdata

!pip install -qU langchain-tavily
from langchain_tavily import TavilySearch

from langchain.tools import tool
import requests

from langchain.agents import create_agent

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 21.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [3]:
!pip install -qU langchain-community pypdf
!pip install -qU langchain langchain-huggingface sentence_transformers
!pip install -qU langchain-chroma
!pip install -q -U \
    "fastapi>=0.133,<1.0" \
    "starlette>=1.3.1,<2.0" \
    "opentelemetry-api==1.42.1" \
    "opentelemetry-sdk==1.42.1" \
    "chromadb" \
    "langchain-chroma"

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import tempfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 37.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.3 MB/s eta 

/tmp/ipykernel_2473/516854712.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [1]:
gemini_api_key=userdata.get("GEMINI_API_KEY")
tavily_api_key=userdata.get("TAVILY_API_KEY")
rapid_api_key=userdata.get("RAPID_API_KEY")


model=init_chat_model(
    "google_genai:gemini-3.6-flash",
    api_key=gemini_api_key
)

skill_demand_tool=TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs based on given skill and location using search-v2."""
    print("calling search job tool")
    print(f"Searching for {skill} jobs in location : {location}")

    url = "https://jsearch.p.rapidapi.com/search-v2"

    querystring = {
        "query": f"{skill} jobs in {location}",
        "num_pages": "1",
        "country": "in",
        "employment_types": "FULLTIME,INTERN",
        "job_requirements": "under_3_years_experience,no_experience",
        "date_posted": "all",
    }

    headers = {
        "x-rapidapi-key": rapid_api_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
        "Content-Type": "application/json",
    }

    response = requests.get(url, headers=headers, params=querystring)
    response.raise_for_status()

    payload = response.json()

    # search-v2 places listings under data -> jobs
    jobs = payload.get("data", {}).get("jobs", [])
    print(f"found {len(jobs)} jobs")

    result = []
    for job in jobs:
        if isinstance(job, dict):
            # Prefer city/state, fall back to general job_location or country
            loc = (
                job.get("job_city")
                or job.get("job_location")
                or job.get("job_country")
                or "Not specified"
            )

            result.append(
                {
                    "title": job.get("job_title", ""),
                    "company": job.get("employer_name", ""),
                    "location": loc,
                    "apply_link": job.get("job_apply_link", ""),
                }
            )

    return result

# Cache one vector store per uploaded resume path, so re-running
# the tool doesn't re-embed and duplicate chunks in Chroma.
_resume_stores = {}

def _get_or_build_resume_store(resume_path: str):
    if resume_path in _resume_stores:
        return _resume_stores[resume_path]

    loader = PyPDFLoader(resume_path)
    doc = loader.load()

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    splits = splitter.split_documents(doc)

    # unique collection per resume so different students' data don't mix
    collection_name = "resume_" + str(abs(hash(resume_path)))
    store = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model,
        persist_directory="./chroma_resume_db",
    )
    store.add_documents(documents=splits)

    _resume_stores[resume_path] = store
    return store


@tool
def match_resume_to_skill(resume_path: str, target_skill: str) -> dict:
    """Given a path to a student's resume PDF and a target skill (e.g. 'Generative AI'),
    retrieve the most relevant resume sections and identify how well the resume
    already covers that skill, plus what's missing."""
    print(f"Matching resume at {resume_path} against skill: {target_skill}")

    store = _get_or_build_resume_store(resume_path)

    query = f"experience, projects, or coursework related to {target_skill}"
    retrieved_docs = store.similarity_search(query, k=4)

    if not retrieved_docs:
        return {"match_summary": "No relevant resume content found.", "sources": []}

    context = "\n\n".join(
        f"[Page {d.metadata.get('page', '?')}]\n{d.page_content}"
        for d in retrieved_docs
    )

    system_message = (
        "You are a career coach. You are given excerpts from a student's resume "
        f"and a target skill: '{target_skill}'.\n"
        "Using ONLY the resume excerpts below, do two things:\n"
        "1. Summarize what relevant experience/projects the student already has.\n"
        "2. List 2-3 concrete skill gaps they should fill to be competitive.\n"
        "Do not invent experience that isn't in the excerpts.\n\n"
        f"Resume excerpts:\n{context}"
    )

    response = model.invoke([
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Assess my fit for {target_skill} roles."},
    ])

    return {
        "match_summary": response.content,
        "sources": [d.metadata for d in retrieved_docs],
    }

system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills
- match_resume_to_skill: If the student has provided a resume PDF path, use this to assess how well their resume matches a target skill and what gaps to fill

Help the student by researching the skill they ask about, checking their resume fit if available, and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""

agent = create_agent(
    model=model,
    tools=[skill_demand_tool, search_jobs, match_resume_to_skill],
    system_prompt=system_prompt,
    debug=True
)
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})

print(response["messages"][-1].content[0]['text'])

NameError: name 'userdata' is not defined

Adding memory agent